[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/notebooks/visualization_yt.ipynb)

# Visualizing OpenImpala Results with yt

OpenImpala uses the [AMReX](https://amrex-codes.github.io/amrex/) backend, which means its output plotfiles are natively supported by [yt](https://yt-project.org/) — a Python package designed for out-of-core analysis of large volumetric datasets.

This notebook demonstrates how to load and visualize OpenImpala results directly in Jupyter, without loading the entire dataset into RAM. This is critical for multi-billion voxel domains that would otherwise trigger Out-of-Memory errors.

**What you will learn:**
1. Run an OpenImpala solve with plotfile output enabled.
2. Load an AMReX plotfile with `yt.load()`.
3. Create 2D slice plots of the solution field.
4. Create 1D profile plots (e.g., average potential along an axis).
5. Extract numerical data for custom analysis.

**Prerequisites:** None — this notebook generates its own data.

## 0. Install Dependencies

In [ ]:
import subprocess, sys

def _has_gpu():
    try:
        subprocess.check_output(["nvidia-smi"], stderr=subprocess.DEVNULL)
        return True
    except (FileNotFoundError, subprocess.CalledProcessError):
        return False

_extras = "yt matplotlib porespy"
if _has_gpu():
    print("GPU detected — installing openimpala-cuda")
    !pip install -q openimpala-cuda {_extras}
else:
    print("No GPU detected — installing openimpala (CPU)")
    !pip install -q openimpala {_extras}

## 1. Generate a Plotfile with OpenImpala

We create a synthetic porous microstructure using PoreSpy, then run the
TortuosityHypre solver with `write_plotfile=True` to produce an AMReX
plotfile on disk.

In [ ]:
import os
import numpy as np
import porespy as ps
import openimpala as oi
from openimpala import core as oic

print(f"OpenImpala version {oi.__version__}")

session = oi.Session()
session.__enter__()

# Generate a 100³ synthetic microstructure
np.random.seed(42)
micro = ps.generators.blobs(
    shape=[100, 100, 100], porosity=0.50, blobiness=1.5,
).astype(np.int32)

print(f"Microstructure: {micro.shape}, porosity={micro.mean():.3f}")

# Compute volume fraction for the solver constructor
vf = oi.volume_fraction(micro, phase=1)

# Wrap the array and run the solver with plotfile output
out_dir = "./plotfiles"
os.makedirs(out_dir, exist_ok=True)

img = oi.VoxelImage.from_numpy(micro.copy(), max_grid_size=64)
solver = oic.TortuosityHypre(
    img,
    vf.fraction,               # volume fraction
    1,                         # phase ID
    oic.Direction.Z,           # solve direction
    oic.SolverType.FlexGMRES,  # HYPRE solver
    out_dir,                   # results directory
    0.0, 1.0,                  # Dirichlet BCs (V_lo, V_hi)
    1,                         # verbose
    True                       # write_plotfile
)

tau = solver.value()
print(f"Tortuosity (Z): {tau:.4f}")
print(f"Converged: {solver.solver_converged}, Iterations: {solver.iterations}")

## 2. Load the Plotfile with yt

yt auto-detects AMReX plotfile format. The solver wrote its output to
`plotfiles/tortuosity_solution_2` (direction Z = axis index 2).

In [ ]:
import yt
import glob

# Suppress verbose yt logging
yt.funcs.mylog.setLevel(40)

# Find the plotfile directory written by the solver
PLOTFILE = "./plotfiles/tortuosity_solution_2"
if not os.path.isdir(PLOTFILE):
    plotfile_dirs = sorted(d for d in glob.glob("./plotfiles/*") if os.path.isdir(d))
    PLOTFILE = plotfile_dirs[-1]

ds = yt.load(PLOTFILE)

print(f"Plotfile:          {PLOTFILE}")
print(f"Domain dimensions: {ds.domain_dimensions}")
print(f"Domain left edge:  {ds.domain_left_edge}")
print(f"Domain right edge: {ds.domain_right_edge}")
print()
print("Available fields:")
for field in sorted(ds.field_list):
    print(f"  {field}")

OpenImpala plotfiles contain three fields:
- `('boxlib', 'solution_potential')` — the solved steady-state concentration/potential field
- `('boxlib', 'phase_id')` — the segmented phase map (0 = pore, 1 = solid)
- `('boxlib', 'active_mask')` — binary mask of voxels included in the solve (connected pore space)

## 3. 2D Slice Plots

`yt.SlicePlot` generates a 2D cross-section through the 3D domain. This is the most common visualization for transport fields — it shows the concentration gradient across the pore network.

**Key advantage:** yt reads data lazily from disk, so this works on datasets far larger than available RAM.

In [ ]:
# Slice through the centre of the domain, normal to the Y axis
field = ('boxlib', 'solution_potential')

slc = yt.SlicePlot(ds, "y", field)
slc.set_cmap(field, "magma")
slc.set_log(field, False)
slc.set_zlim(field, 0.0, 1.0)
slc.set_colorbar_label(field, "Potential $\\phi$")
slc.annotate_title("Steady-State Diffusion Potential (Y-slice)")
slc.show()

In [ ]:
# Phase map — shows pore (0) vs solid (1)
phase_field = ('boxlib', 'phase_id')

slc_phase = yt.SlicePlot(ds, "y", phase_field)
slc_phase.set_cmap(phase_field, "RdBu")
slc_phase.set_log(phase_field, False)
slc_phase.set_colorbar_label(phase_field, "Phase ID")
slc_phase.annotate_title("Phase Map")
slc_phase.show()

In [ ]:
# Slices at different depths along the flow direction (Z)
for z_frac in [0.25, 0.5, 0.75]:
    center = ds.domain_center.copy()
    center[2] = ds.domain_left_edge[2] + z_frac * (ds.domain_right_edge[2] - ds.domain_left_edge[2])

    slc = yt.SlicePlot(ds, "z", field, center=center)
    slc.set_cmap(field, "magma")
    slc.set_log(field, False)
    slc.set_zlim(field, 0.0, 1.0)
    slc.annotate_title(f"Z = {z_frac:.0%} of domain")
    slc.show()

## 4. 1D Profile Plots

`yt.ProfilePlot` computes bin-averaged quantities along a chosen axis. This is ideal for visualizing how the average potential varies along the flow direction — the gradient of this curve is directly related to the effective diffusivity.

In [ ]:
ad = ds.all_data()

profile = yt.ProfilePlot(
    ad,
    ("index", "z"),                    # bin along Z (the flow direction)
    ("boxlib", "solution_potential"),   # average this field
    weight_field=("index", "ones"),     # simple average
)

profile.set_ylabel(("boxlib", "solution_potential"), "Average Potential $\\langle\\phi\\rangle$")
profile.set_xlabel("Position along Z (flow direction)")
profile.annotate_title("Through-Thickness Potential Profile")
profile.show()

**Interpreting the profile:**
- For a uniform medium with Dirichlet BCs ($\phi=0$ at inlet, $\phi=1$ at outlet), the profile is a straight line.
- For a heterogeneous porous medium, the profile deviates from linearity. Steeper gradients indicate regions of higher resistance (lower local diffusivity).
- The overall slope relates to the effective diffusivity: $D_{\text{eff}} = \text{flux} / |\nabla\phi|$.

## 5. Extracting Numerical Data

For custom analysis beyond built-in yt plots, extract data into NumPy arrays.

In [ ]:
import matplotlib.pyplot as plt

ad = ds.all_data()
prof = yt.create_profile(
    ad,
    ("index", "z"),
    ("boxlib", "solution_potential"),
    n_bins=64,
    weight_field=("index", "ones"),
)

z = np.array(prof.x)
phi_avg = np.array(prof[("boxlib", "solution_potential")])

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.plot(z, phi_avg, 'o-', color='#1f77b4', ms=4, lw=1.5)
ax.set_xlabel("Position along flow direction (Z)", fontweight='bold')
ax.set_ylabel("$\\langle\\phi\\rangle$", fontweight='bold', fontsize=14)
ax.set_title("Through-Thickness Potential Profile")
ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

dphi_dz = np.gradient(phi_avg, z)
print(f"Mean gradient dphi/dz = {np.mean(dphi_dz):.6f}")

In [ ]:
# Extract a 2D slice as a NumPy array (fixed resolution buffer)
slc = ds.slice("y", ds.domain_center[1])

# frb = Fixed Resolution Buffer — rasterizes the AMR data to a uniform grid
frb = slc.to_frb(
    width=ds.domain_width[0],
    resolution=(512, 512),
)

phi_2d = np.array(frb[("boxlib", "solution_potential")])

fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
im = ax.imshow(phi_2d, origin='lower', cmap='magma', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Potential $\\phi$')
ax.set_title("Solution Field (Y-midplane)")
ax.set_xlabel("X")
ax.set_ylabel("Z")
plt.tight_layout()
plt.show()

print(f"Extracted array shape: {phi_2d.shape}")
print(f"Value range: [{phi_2d.min():.4f}, {phi_2d.max():.4f}]")

## 6. Working with Large Datasets

The key advantage of yt is **lazy loading** — it only reads the data needed for the current operation. This means the techniques above work identically on a $100^3$ test domain and a $2000^3$ synchrotron dataset.

### Tips for large datasets

| Technique | Memory Usage | When to Use |
|-----------|-------------|-------------|
| `yt.SlicePlot` | Low — reads one 2D plane | Quick visual inspection |
| `yt.ProjectionPlot` | Medium — integrates along an axis | Overview of field structure |
| `yt.ProfilePlot` | Low — bin-averages on the fly | Quantitative 1D analysis |
| `ds.all_data()` | **High** — reads everything | Only for small datasets or HPC |
| `ds.region(...)` | Configurable | Sub-volume analysis |

### Selecting sub-regions

```python
# Analyse only a sub-volume (avoids loading the full dataset)
left = ds.domain_left_edge + 0.25 * ds.domain_width
right = ds.domain_left_edge + 0.75 * ds.domain_width
region = ds.region(center=0.5*(left+right), left_edge=left, right_edge=right)

# All yt operations work on regions too
profile = yt.ProfilePlot(region, ("index", "z"), ("boxlib", "solution_potential"))
```

### Producing plotfiles from your own data

To run OpenImpala on your own data with plotfile output, either use the Python API (as shown above) or set `write_plotfile = 1` in an inputs file:

```ini
filename = my_sample.tif
phase_id = 1
direction = Z
solver_type = FlexGMRES
write_plotfile = 1
```

## References

- [yt documentation](https://yt-project.org/docs/dev/)
- [yt AMReX frontend](https://yt-project.org/docs/dev/examining/loading_data.html#amrex-boxlib-data)
- [AMReX plotfile format](https://amrex-codes.github.io/amrex/docs_html/Visualization.html)
- [OpenImpala tutorials](../tutorials/)